# Shallow-network approximation experiment

Copy this notebook for your target function. Edit the function and its parameters, then run the cells in order. Shared data, training, evaluation, and baseline code lives in `src/`.

In [ ]:
from pathlib import Path
import sys
from functools import partial
import numpy as np
import pandas as pd

# This lets the imports work whether the notebook opens from the repo or notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.functions import affine, exponential, polynomial, sinusoidal
from src.model import train_model
from src.experiment import generate_data, evaluate_model, regression_baselines

## Choose the target

Replace the example below with your target and its parameters. For example, use `partial(sinusoidal, amplitude=2, offset=1)` to double the wave and shift it up by 1.

In [ ]:
# Swap in your target and tweak its parameters here.
target_name = 'exponential'
target = partial(exponential, amplitude=1.0, rate=1.0, offset=0.0)

# Examples for the other families:
# target_name = 'affine'; target = partial(affine, slope=2.0, intercept=1.0)
# target_name = 'polynomial'; target = partial(polynomial, cubic=1.0, quadratic=0.0, linear=-1.0, constant=0.0)
# target_name = 'sinusoidal'; target = partial(sinusoidal, amplitude=1.0, frequency=2 * np.pi, phase=0.0, offset=0.0)

In [ ]:
# Make the same train/validation/test split for the target in this notebook.
domain = (-1.0, 1.0)
data = generate_data(
    target, n_train=1000, n_validation=1000, n_test=10000,
    domain=domain, seed=21,
)

x_train, y_train = data['train']
x_validation, y_validation = data['validation']
x_test, y_test = data['test']
print(f'train: {len(x_train)}, validation: {len(x_validation)}, test: {len(x_test)}')

In [ ]:
# For each width, train several random starts, test the selected model, and save its scores.
widths = [1, 2, 4, 8, 16, 32, 64, 128, 256]
seeds = range(21, 31)
rows = []
fitted_models = {}

for width in widths:
    fit = train_model(
        x_train, y_train, x_validation, y_validation,
        width=width, seeds=seeds,
    )
    test_error = evaluate_model(fit['parameters'], x_test, y_test)
    fitted_models[width] = fit
    rows.append({
        'target': target_name,
        'model': 'sigmoid network',
        'width': width,
        'validation_max_error': fit['validation_max_error'],
        'gradient_norm': fit['gradient_norm'],
        'test_max_error': test_error,
    })

for baseline in regression_baselines(x_train, y_train, x_test, y_test):
    rows.append({
        'target': target_name, 'width': '', 'validation_max_error': '', 'gradient_norm': '',
        **baseline,
    })

results = pd.DataFrame(rows)
results

In [ ]:
output_dir = PROJECT_ROOT / 'results'
output_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(output_dir / f'{target_name}_metrics.csv', index=False)

In [ ]:
# Nina: add visualization code here, using data, fitted_models, and results.